In [1]:
import numpy as np
import random

In [6]:
# Define state and actions
states= np.arange(0,101,10)
actions= ['FULL','STOP']
print('States:', states)
print('Actions:', actions)

States: [  0  10  20  30  40  50  60  70  80  90 100]
Actions: ['FULL', 'STOP']


In [7]:
Q= np.zeros((len(states), len(actions)))

alpha= 0.1
gamma= 0.9
epsilon= 0.2
episodes= 300

print('Q-table shape:', Q.shape)

Q-table shape: (11, 2)


In [11]:
def get_reward(level, action):
    if 40 <= level <= 70:
        reward = 10   # ideal range
    else:
        reward = -10   # too low/high
    if action == 'FILL':
        reward -= 10   # overflow risk
    if action == 'STOP':
        reward -= 10   # empty risk
    return reward
print('Reward example (level=60), FILL', get_reward(60,'FILL'))

Reward example (level=60), FILL 0


In [12]:
def next_level(level, action):
    if action == 'FILL':
        level += random.choice([5,10,15])
    else:
        level -= random.choice([5,10,15])
    return int(np.clip(level, 0, 100))

print('Next Level Example:', next_level(50, 'Fill'))

Next Level Example: 40


In [17]:
for ep in range(episodes):
    level= random.choice(states)
    for _ in range(15):
        if random.uniform(0, 1) < epsilon:
            action = random.choice(actions)
        else:
            action = actions[np.argmax(Q[level // 10])]
        next_state = next_level(level, action)
        reward = get_reward(next_state, action)
        
        a = actions.index(action)
        best_next = np.max(Q[next_state // 10])
        Q[level // 10, a] += alpha * (reward + gamma * best_next - Q[level // 10, a])
        level = next_state
print('Training Completed')

Training Completed


In [18]:
try:
    level = int(input('Enter starting water level (0-100): '))
    if level < 0 or level > 100:
        raise ValueError('Water level out of range!')
except ValueError as e:
    print(e)
    level=50
    print('Default level set to 50%.')

print(f'\nStarting with leve: {level}%')
print('Simulating for 10 steps:\n')

for step in range(10):
    action = actions[np.argmax(Q[level // 10])]
    print(f'Step {step+1}: Level={level}% → Action={action}')
    level = next_level(level, action)

print('\nSimulation complete. Water tank control finished.')
    

Enter starting water level (0-100):  80



Starting with leve: 80%
Simulating for 10 steps:

Step 1: Level=80% → Action=FULL
Step 2: Level=75% → Action=FULL
Step 3: Level=65% → Action=FULL
Step 4: Level=50% → Action=FULL
Step 5: Level=45% → Action=FULL
Step 6: Level=35% → Action=FULL
Step 7: Level=20% → Action=FULL
Step 8: Level=15% → Action=FULL
Step 9: Level=5% → Action=FULL
Step 10: Level=0% → Action=FULL

Simulation complete. Water tank control finished.


# new exercise

In [1]:
import numpy as np 
import random    

In [53]:
# Step 2: Define States and Actions
# Traffic states are represented by their index (0-4)
# 0: Empty, 1: Light, 2: Moderate, 3: Heavy, 4: Very Heavy
states = np.arange(0, 5) # [0, 1, 2, 3, 4]
actions = ['GREEN', 'RED']
print('States:', states)
print('Actions:', actions)

States: [0 1 2 3 4]
Actions: ['GREEN', 'RED']


In [4]:
# Step 3: Initialize the Q-Table and Hyperparameters 

# Q-Table: 5 states (rows) x 2 actions (columns)
Q = np.zeros((len(states), len(actions)))

# Hyperparameters
alpha = 0.1      # Learning rate
gamma = 0.9      # Discount factor
epsilon = 0.2    # Exploration rate
episodes = 300   # Number of training episodes

print('Q-table shape:', Q.shape)

NameError: name 'states' is not defined

In [55]:
#  Step 4: Design the Reward Function 

def get_reward(traffic_level, action):
    # traffic_level is the state index (0-4)
    
    # 3. Heavy traffic & 4. Very Heavy traffic (Grouped)
    if traffic_level >= 3:
        if action == 'GREEN':
            return 10   # Good: clears congestion
        else: # action == 'RED'
            return -10  # Bad: creates jams
            
    # 0. Empty road
    elif traffic_level == 0:
        if action == 'RED':
            return 5    # Good: Saves energy
        else: # action == 'GREEN'
            return -5   # Wastes power
            
    # 1. Light traffic and 2. Moderate traffic
    else: 
        return 1       # Neutral/Slightly positive

print('Reward example (Heavy Traffic=3, GREEN):', get_reward(3,'GREEN'))

Reward example (Heavy Traffic=3, GREEN): 10


In [56]:
#  Step 5: Define Environment Dynamics 

def next_traffic(current_level):
    """Simulates how traffic changes: randomly +/- 1 level."""
    # Change by -1, 0, or 1
    change = random.choice([-1, 0, 1])
    next_level = current_level + change
    
    # Ensure traffic level stays between 0 and 4
    return int(np.clip(next_level, 0, len(states) - 1))

print('Next Traffic Example (Current=2):', next_traffic(2))

Next Traffic Example (Current=2): 2


In [57]:
#  Step 6: Train the Agent 

print('\n--- Starting Q-Learning Training ---')

# We'll use 1 decision step per episode for simplicity, similar to your trainer's style
for ep in range(episodes):
    # Start from a random traffic level
    current_state = random.choice(states)
    
    # Action selection: epsilon-greedy
    if random.uniform(0, 1) < epsilon:
        # Explore: Choose a random action
        action = random.choice(actions)
    else:
        # Exploit: Choose the best action from Q-table
        action = actions[np.argmax(Q[current_state])]
        
    # Get environment feedback
    next_state = next_traffic(current_state)
    reward = get_reward(next_state, action)
    
    # Q-table update
    a = actions.index(action)
    best_next = np.max(Q[next_state])
    
    # Q(s, a) = Q(s, a) + alpha * [r + gamma * max_a' Q(s', a') - Q(s, a)]
    Q[current_state, a] += alpha * (reward + gamma * best_next - Q[current_state, a])
    
print('Training Completed')


--- Starting Q-Learning Training ---
Training Completed


In [58]:
# Step 7: Test the Learned Traffic Light Controller 

print('\n--- Testing the Learned Controller ---')

try:
    start_level = int(input(f'Enter starting traffic level (0={actions[0]} to 4={actions[1]}): '))
    if start_level < 0 or start_level >= len(states):
        raise ValueError('Traffic level out of range!')
except ValueError as e:
    print(e)
    start_level = 2 # Default to Moderate
    print('Default level set to 2 (Moderate).')

current_traffic = start_level

print(f'\nStarting traffic: {current_traffic}')
print('Simulating for 10 steps:\n')

for step in range(10):
    # Choose the best action (exploitation)
    action = actions[np.argmax(Q[current_traffic])]
    
    # Print state and action
    print(f'Step {step+1}: Traffic Level={current_traffic} → Action={action}')
    
    # Move to the next traffic level
    current_traffic = next_traffic(current_traffic)

print('\nSimulation complete. Smart traffic light decision finished.')


--- Testing the Learned Controller ---


Enter starting traffic level (0=GREEN to 4=RED):  2



Starting traffic: 2
Simulating for 10 steps:

Step 1: Traffic Level=2 → Action=GREEN
Step 2: Traffic Level=1 → Action=RED
Step 3: Traffic Level=1 → Action=RED
Step 4: Traffic Level=2 → Action=GREEN
Step 5: Traffic Level=2 → Action=GREEN
Step 6: Traffic Level=1 → Action=RED
Step 7: Traffic Level=0 → Action=RED
Step 8: Traffic Level=1 → Action=RED
Step 9: Traffic Level=2 → Action=GREEN
Step 10: Traffic Level=2 → Action=GREEN

Simulation complete. Smart traffic light decision finished.


In [52]:
# Step 8: Analyze the Behavior (Discussion points) 

print('\n### Analysis of the Learned Behavior (Q-Table Insights)')
print(f"Q-Table (Scores):\n{Q.round(2)}")

# Check heavy traffic state (index 3)
heavy_action = actions[np.argmax(Q[3])]
print(f"\n1. Heavy Traffic (State 3/4): The system chooses **{heavy_action}** because the Q-score for GREEN is much higher, reflecting the +10 reward for clearing congestion.")

# Check empty road state (index 0)
empty_action = actions[np.argmax(Q[0])]
print(f"2. Empty Road (State 0): The system chooses **{empty_action}** because the Q-score for RED is positive, reflecting the +5 reward for saving energy.")

# Check moderate traffic state (index 2)
moderate_action = actions[np.argmax(Q[2])]
print(f"3. Moderate Traffic (State 2): The choice is often less extreme, but it generally favors the action that minimizes risk or leads to the most 'stable' future state.")

print("\n**Q-table change:** Successful state-action pairs (like Heavy+GREEN) accumulate high positive scores, making them the preferred choice for future decisions.")


### Analysis of the Learned Behavior (Q-Table Insights)
Q-Table (Scores):
[[ 1.54 12.54]
 [ 2.23 14.33]
 [19.79  5.11]
 [25.67  4.11]
 [35.33  2.9 ]]

1. Heavy Traffic (State 3/4): The system chooses **GREEN** because the Q-score for GREEN is much higher, reflecting the +10 reward for clearing congestion.
2. Empty Road (State 0): The system chooses **RED** because the Q-score for RED is positive, reflecting the +5 reward for saving energy.
3. Moderate Traffic (State 2): The choice is often less extreme, but it generally favors the action that minimizes risk or leads to the most 'stable' future state.

**Q-table change:** Successful state-action pairs (like Heavy+GREEN) accumulate high positive scores, making them the preferred choice for future decisions.
